## markdown, stats

In [ ]:
import os
from os.path import join
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
from statsmodels.stats.anova import AnovaRM
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

import warnings
warnings.filterwarnings("ignore")

manuscript_dir = r'C:\Users\Radovan\OneDrive\Radboud\Studentships\Jordy Thielen\Manuscript'
m_ica_dir = join(manuscript_dir, 'data', 'anova', 'ica')
m_noica_dir = join(manuscript_dir, 'data', 'anova', 'noica')

def load_decoding_results(npz_path):
    with np.load(npz_path) as data:
        epoch_bins_loaded = data['subject']
        mean_acc = data['accuracies']
        #se_acc = data['ses']
    return epoch_bins_loaded, mean_acc

epoch_bins = np.linspace(5, 80, 16, dtype=int)
task = 'covert'


# Full paths
data1_path = os.path.join(m_ica_dir, "covert_lda_alphaCSP_dec_64_ica.npz")
data2_path = os.path.join(m_noica_dir, "covert_lda_alphaCSP_dec_64_noica.npz" )
data3_path = os.path.join(m_ica_dir, "covert_lda_p300_dec_64_ica.npz")
data4_path = os.path.join(m_noica_dir, "covert_lda_p300_dec_64_noica.npz" )
data5_path = os.path.join(m_ica_dir, "covert_lda_rcca_dec_64_ica.npz" )
data6_path = os.path.join(m_noica_dir, "covert_lda_rcca_dec_64_noica.npz" )
# --...---...---...---...---...---...---...---...---...---...---...---...---...-
subj1, mean_acc1 = load_decoding_results(data1_path)
subj2, mean_acc2 = load_decoding_results(data2_path)
subj3, mean_acc3 = load_decoding_results(data3_path)
subj4, mean_acc4 = load_decoding_results(data4_path)
subj5, mean_acc5    = load_decoding_results(data5_path)
subj6, mean_acc6 = load_decoding_results(data6_path)
# --...---...---...---...---...---...---...---...---...---...---...---...---...-
data = {
    'Alpha w/ ICA': {
        'mean': mean_acc1,

    },
    'Alpha wo/ ICA': {
        'mean': mean_acc2,
 
    },
    'P300 w/ ICA': {
        'mean': mean_acc3,

        
    },
    'P300 wo/ ICA': {
        'mean': mean_acc4,

        

},
    'cVEP w/ ICA': {
        'mean': mean_acc5,

        
    },
    'cVEP wo/ ICA': {
        'mean': mean_acc6,

        

}
}

In [4]:
# Build long Dataframe for ANOVA
import pandas as pd
rows = []
n_subjects = subj1.size
for cond_label, acc in data.items():
    strategy = cond_label.split()[0]   # “P300”, “Alpha”, “cVEP”
    ica_flag = 'With ICA' if 'w/ ICA' in cond_label else 'No ICA'
    for subj in range(n_subjects):
        rows.append({
            'subject': subj,
            'strategy': strategy,
            'ICA': ica_flag,
            'accuracy': np.mean(acc['mean'][subj])
        })
df = pd.DataFrame(rows)

In [ ]:

# Two-way RM ANOVA with eta-squared
rmaov = AnovaRM(df, depvar='accuracy', subject='subject',
                within=['strategy', 'ICA']).fit()
anova_table = rmaov.anova_table.copy()
anova_table['eta_sq'] = (
    anova_table['Num DF'] * anova_table['F Value']
) / (
    anova_table['Num DF'] * anova_table['F Value'] + anova_table['Den DF']
)

print("ANOVA table with partial eta-squared:")
print(anova_table)

# Collapse over ICA for post-hoc tests
df_strat = (
    df
    .groupby(['subject','strategy'])['accuracy']
    .mean()
    .reset_index()
)

# One-sided paired t-tests and Bonferroni correction
hypotheses = [
    ('P300',  'Alpha'),   # H1: P300 > Alpha
    ('Alpha', 'cVEP'),    # H1: Alpha > cVEP
    ('P300',  'cVEP')     # H1: P300 > cVEP
]

results = []
for greater, lesser in hypotheses:
    x = df_strat.query("strategy == @greater").sort_values('subject')['accuracy']
    y = df_strat.query("strategy == @lesser").sort_values('subject')['accuracy']
    tstat, p_two_sided = ttest_rel(x, y)
    # convert to one-sided p-value
    p_one_sided = p_two_sided / 2 if tstat > 0 else 1 - p_two_sided / 2
    results.append((f"{greater} > {lesser}", tstat, p_one_sided))

# Bonferroni correction
pvals = [r[2] for r in results]
rej, pvals_bonf, _, _ = multipletests(pvals, alpha=0.05, method='bonferroni')

# Display post-hoc results
print("\nPost-hoc Bonferroni-corrected one-sided t-tests:")
for (hyp, tval, p1), sig, p_corr in zip(results, rej, pvals_bonf):
    print(f"{hyp}: t = {tval:.3f}, p(one-sided) = {p1:.4f}, " +
          f"p_bonf = {p_corr:.4f}, significant = {sig}")


ANOVA table with partial eta-squared:
                F Value  Num DF  Den DF        Pr > F    eta_sq
strategy      84.906948     2.0    56.0  1.107251e-17  0.752008
ICA            0.009391     1.0    28.0  9.234896e-01  0.000335
strategy:ICA   4.652738     2.0    56.0  1.351064e-02  0.142492

Post-hoc Bonferroni-corrected one-sided t-tests:
P300 > Alpha: t = 4.144, p(one-sided) = 0.0001, p_bonf = 0.0004, significant = True
Alpha > cVEP: t = 6.919, p(one-sided) = 0.0000, p_bonf = 0.0000, significant = True
P300 > cVEP: t = 19.166, p(one-sided) = 0.0000, p_bonf = 0.0000, significant = True
